# Simple Video Inference Pipeline - Interactive Tutorial

This notebook demonstrates how to use the new SimpleVideoDataset pipeline for running deepfake detection inference on new videos without metadata.

## 1. Setup

Import the necessary modules.

In [ ]:
import torch
from dataset import SimpleVideoDataset, create_simple_dataloader
from model.batfd_plus import BatfdPlus
from model.batfd import Batfd
import os

## 2. Basic Usage - Single Video

Process a single video file.

In [ ]:
# Path to your video (UPDATE THIS)
video_path = "path/to/your/video.mp4"

# Create dataset
dataset = SimpleVideoDataset(
    video_paths=video_path,
    frame_padding=512,
    fps=25,
    return_file_name=True
)

# Get the video data
video, audio, n_frames, filename = dataset[0]

print(f"Video shape: {video.shape}")  # (C, T, H, W) = (3, 512, 96, 96)
print(f"Audio shape: {audio.shape}")  # (64, 2048) - log mel spectrogram
print(f"Number of frames: {n_frames}")
print(f"Filename: {filename}")

## 3. Load a Trained Model

Load your pre-trained BATFD or BATFD+ model.

In [ ]:
# Path to your model checkpoint (UPDATE THIS)
checkpoint_path = "path/to/checkpoint.ckpt"

# Load BATFD+ model
model = BatfdPlus.load_from_checkpoint(checkpoint_path)
model.eval()

# Move to GPU if available
device = "cuda" if torch.cuda.is_available() else "cpu"
model = model.to(device)

print(f"Model loaded on device: {device}")

## 4. Run Inference on Single Video

In [ ]:
# Prepare inputs
video_input = video.unsqueeze(0).to(device)  # Add batch dimension
audio_input = audio.unsqueeze(0).to(device)

# Run inference
with torch.no_grad():
    outputs = model(video_input, audio_input)

# Unpack outputs (BATFD+ model)
fusion_bm, fusion_start, fusion_end, visual_bm, visual_start, visual_end, audio_bm, audio_start, audio_end = outputs

print(f"Fusion boundary map shape: {fusion_bm.shape}")
print(f"Maximum fusion score: {fusion_bm.max().item():.4f}")
print(f"Maximum visual score: {visual_bm.max().item():.4f}")
print(f"Maximum audio score: {audio_bm.max().item():.4f}")

## 5. Process Multiple Videos from Directory

Use the DataLoader to process multiple videos efficiently.

In [ ]:
# Path to directory containing videos (UPDATE THIS)
video_dir = "path/to/videos/"

# Create dataloader
dataloader = create_simple_dataloader(
    video_paths=video_dir,
    batch_size=2,
    num_workers=0
)

print(f"Found {len(dataloader.dataset)} videos to process")

## 6. Batch Processing with Progress Tracking

In [ ]:
from tqdm import tqdm

results = []

with torch.no_grad():
    for batch_idx, batch in enumerate(tqdm(dataloader, desc="Processing videos")):
        video, audio, n_frames, filenames = batch
        
        # Move to device
        video = video.to(device)
        audio = audio.to(device)
        
        # Run model
        outputs = model(video, audio)
        fusion_bm = outputs[0]  # Get fusion boundary map
        
        # Store results for each video in batch
        for i in range(len(filenames)):
            result = {
                'file': filenames[i],
                'n_frames': n_frames[i].item(),
                'max_score': fusion_bm[i].max().item(),
                'boundary_map': fusion_bm[i].cpu()
            }
            results.append(result)

print(f"\nProcessed {len(results)} videos")

## 7. Analyze Results

In [ ]:
import pandas as pd

# Create summary dataframe
summary = pd.DataFrame([
    {
        'filename': os.path.basename(r['file']),
        'frames': r['n_frames'],
        'max_score': r['max_score']
    }
    for r in results
])

print("\nResults Summary:")
print(summary)

print(f"\nAverage max score: {summary['max_score'].mean():.4f}")
print(f"Highest max score: {summary['max_score'].max():.4f}")

## 8. Visualize Boundary Map (Optional)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Visualize first result
if len(results) > 0:
    result = results[0]
    bm = result['boundary_map'].numpy()
    
    plt.figure(figsize=(12, 6))
    plt.imshow(bm, aspect='auto', cmap='hot')
    plt.colorbar(label='Score')
    plt.xlabel('Frame')
    plt.ylabel('Duration')
    plt.title(f"Boundary Map: {os.path.basename(result['file'])}")
    plt.tight_layout()
    plt.show()
    
    print(f"Max score location: {np.unravel_index(bm.argmax(), bm.shape)}")

## 9. Save Results

In [ ]:
import os
from pathlib import Path

# Create output directory
output_dir = "output/notebook_inference"
Path(output_dir).mkdir(parents=True, exist_ok=True)

# Save each result
for result in results:
    filename = os.path.basename(result['file']).replace('.mp4', '.pt')
    output_path = os.path.join(output_dir, filename)
    
    # Save boundary map
    torch.save({
        'boundary_map': result['boundary_map'],
        'n_frames': result['n_frames'],
        'max_score': result['max_score']
    }, output_path)
    
    print(f"Saved: {output_path}")

# Save summary as CSV
summary.to_csv(os.path.join(output_dir, 'summary.csv'), index=False)
print(f"\nSummary saved to: {os.path.join(output_dir, 'summary.csv')}")

## 10. Process Specific Videos

In [ ]:
# List of specific videos to process
specific_videos = [
    "video1.mp4",
    "video2.mp4",
    "video3.mp4"
]

# Create dataset
dataset = SimpleVideoDataset(
    video_paths=specific_videos,
    return_file_name=True
)

# Process each video
for idx in range(len(dataset)):
    video, audio, n_frames, filename = dataset[idx]
    
    # Run inference
    with torch.no_grad():
        video_input = video.unsqueeze(0).to(device)
        audio_input = audio.unsqueeze(0).to(device)
        outputs = model(video_input, audio_input)
        fusion_bm = outputs[0]
    
    print(f"[{idx+1}/{len(dataset)}] {os.path.basename(filename)}: max_score={fusion_bm.max().item():.4f}")

## Summary

This notebook demonstrated:
1. ✅ Loading single videos with SimpleVideoDataset
2. ✅ Loading and running a trained model
3. ✅ Processing multiple videos from a directory
4. ✅ Batch processing with DataLoader
5. ✅ Analyzing and visualizing results
6. ✅ Saving outputs

For more information, see:
- `SIMPLE_PIPELINE_README.md` - Quick reference
- `SIMPLE_INFERENCE_GUIDE.md` - Comprehensive guide
- `examples/simple_video_inference_example.py` - Python examples